# BOLD Mortality Prediction — First-Pass Pipeline

**Project:** COMFORT (personalised illness monitoring) — early pipeline development.

**What this does:** using only labs/vitals available at a *single point in time*, predict
whether an ICU patient died during their hospital admission — and explain *which* markers
drive that prediction, rather than just producing a number.

**Scope of this pass:** a simple, explainable baseline. Logistic regression + Random Forest,
SHAP for explanation, and a lightweight confounder check (does a marker's link to mortality
survive once we account for how sick the patient already was?).

**Deliberate trade-off:** the wider project's end goal is a causal framework plus a
transformer model, with an LLM layered in later. None of that is here. Step 7's confounder
check is a deliberately simple stand-in for real causal inference (no matching, IPTW, or DAGs)
— it is a first step towards separating correlation from causation, not the finished article.

## Step 0 — Data access

Real BOLD is a **credentialed PhysioNet dataset**
(https://physionet.org/content/blood-gas-oximetry/1.0/). Access needs a PhysioNet account,
completed ethics/CITI training, and a signed Data Use Agreement — still in progress.

So this pipeline currently runs on a **synthetic placeholder file** with the same column
names and realistic-looking value ranges as real BOLD, but 100% fabricated values. The point
is to get the pipeline working end-to-end, so that switching to real data is a one-line
change to `DATA_PATH` below and nothing else.

In [1]:
# ==========================================================================
#  CHANGE THIS ONE LINE when real PhysioNet access comes through:
#      DATA_PATH = "bold_dataset.csv"
#  Nothing else in this notebook needs to change.
# ==========================================================================
DATA_PATH = "synthetic_bold_dataset.csv"

# Anything that isn't the real BOLD file is synthetic, so the warning below
# switches itself off automatically once DATA_PATH points at the real data.
IS_SYNTHETIC = DATA_PATH != "bold_dataset.csv"

if IS_SYNTHETIC:
    print("=" * 78)
    print("  RUNNING ON SYNTHETIC DATA")
    print("  Every number, plot and SHAP ranking below is FABRICATED.")
    print("  These are NOT real findings - this only proves the pipeline works.")
    print("  Do not quote any result below in a report or to a supervisor.")
    print("=" * 78)
else:
    print("=" * 78)
    print("  RUNNING ON REAL BOLD DATA - results below are genuine.")
    print("=" * 78)

  RUNNING ON SYNTHETIC DATA
  Every number, plot and SHAP ranking below is FABRICATED.
  These are NOT real findings - this only proves the pipeline works.
  Do not quote any result below in a report or to a supervisor.


## Step 1 — Load and look

Load the file and get a feel for it before touching anything: how many rows and columns,
what type each column is, and how much of each column is actually filled in.

**Nothing is dropped here.** This step only reports. Deciding what to throw away comes later
(Step 4), once we've seen the picture.

In [2]:
import pandas as pd

# Show all rows when we print the missingness table - there are ~70 columns and
# pandas would otherwise hide the middle of it behind "...".
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 120)

df = pd.read_csv(DATA_PATH)

print(f"Loaded: {DATA_PATH}")
print(f"Rows (patient records): {df.shape[0]:,}")
print(f"Columns:                {df.shape[1]}")

Loaded: synthetic_bold_dataset.csv
Rows (patient records): 2,000
Columns:                71


In [3]:
# What type is each column? Numbers we can model directly; "object" means text
# (e.g. race_ethnicity, source_db) and would need encoding before use.
print("Column types:")
print(df.dtypes.value_counts())
print()
print("Non-numeric (text) columns:")
text_cols = df.select_dtypes(exclude="number").columns.tolist()
print(text_cols if text_cols else "  (none)")

Column types:
float64    51
int64      18
str         2
Name: count, dtype: int64

Non-numeric (text) columns:
['source_db', 'race_ethnicity']


In [4]:
# How much of each column is actually filled in?
# In real ICU data some labs are ordered for nearly everyone (e.g. basic bloods)
# while specialist tests are only ordered when a doctor already suspects a problem -
# so missingness varies enormously between columns.
missing = pd.DataFrame({
    "n_missing": df.isna().sum(),
    "pct_missing": (df.isna().mean() * 100).round(1),
})
missing = missing.sort_values("pct_missing", ascending=False)

print("Missingness per column (worst first):")
print(missing.to_string())

Missingness per column (worst first):
                                 n_missing  pct_missing
others_ck_mb                          1109         55.4
others_ck_cpk                          902         45.1
hfp_bilirubin_direct                   806         40.3
others_ld_ldh                          780         39.0
coag_fibrinogen                        687         34.4
hfp_albumin                            669         33.4
hfp_alp                                653         32.6
hfp_bilirubin_total                    615         30.8
hfp_alt                                611         30.6
hfp_ast                                594         29.7
bmp_lactate                            476         23.8
coag_ptt                               413         20.6
coag_inr                               293         14.6
coag_pt                                286         14.3
cbc_rdw                                198          9.9
bmp_bun                                  0          0.0
cbc_rbc   

In [5]:
# Summarise the same thing in buckets, so it's easy to see at a glance how many
# columns are in good shape versus how many are mostly empty.
# The 60% line is where we'll cut in Step 4 - flagged here, not acted on yet.
complete    = missing[missing["pct_missing"] == 0]
usable      = missing[(missing["pct_missing"] > 0) & (missing["pct_missing"] <= 30)]
patchy      = missing[(missing["pct_missing"] > 30) & (missing["pct_missing"] <= 60)]
mostly_empty = missing[missing["pct_missing"] > 60]

print(f"Fully complete (0% missing):        {len(complete):>3} columns")
print(f"Usable (1-30% missing):             {len(usable):>3} columns")
print(f"Patchy (31-60% missing):            {len(patchy):>3} columns")
print(f"Mostly empty (>60% missing):        {len(mostly_empty):>3} columns  <- candidates to drop in Step 4")
print()

if len(mostly_empty):
    print("Columns over the 60% missing line:")
    print(mostly_empty.to_string())
else:
    print("No column is over the 60% missing line.")

Fully complete (0% missing):         56 columns
Usable (1-30% missing):               6 columns
Patchy (31-60% missing):              9 columns
Mostly empty (>60% missing):          0 columns  <- candidates to drop in Step 4

No column is over the 60% missing line.
